## Objective
The objective of this notebook is to collect real-time commodity market data, clean and normalize it, and prepare a high-quality, analysis-ready dataset suitable for visualization and machine learning.

## Library Imports

We import essential libraries for:

HTTP requests (requests)

HTML parsing (BeautifulSoup)

Data manipulation (pandas, numpy)

Time handling (datetime, time)

File management (os)

Randomization (random) for polite scraping

A pool of rotating User-Agents is configured to mimic real browsers and reduce scraping blocks.

In [2]:
# Import requests library for making HTTP requests to websites
import requests

# Import BeautifulSoup for parsing and navigating HTML content
from bs4 import BeautifulSoup

# Import pandas for data manipulation and CSV export
import pandas as pd

# Import time for adding delays between requests
import time

# Import random for randomizing delays and user agents
import random

# Import datetime for timestamping our scraped data
from datetime import datetime

# Import os for file and directory management
import os



print("Libraries imported successfully")

Libraries imported successfully


In [3]:
# Define a list of user agents representing different browsers and operating systems
# This helps distribute requests and appears more natural to web servers
USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:89.0) Gecko/20100101 Firefox/89.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.1 Safari/605.1.15'
]

def get_random_user_agent():
    """
    Select and return a random user agent from the predefined list.
    
    Returns:
        str: A random user agent string
    """
    return random.choice(USER_AGENTS)

print(f"User-agent pool configured with {len(USER_AGENTS)} options")
print(f"Sample user agent: {get_random_user_agent()[:50]}...")

User-agent pool configured with 5 options
Sample user agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) Ap...


## Web Scraping from TradingEconomics

Scrapes the TradingEconomics commodities page

Dynamically extracts:

Commodity category

Commodity name

Unit and currency

Price and percentage changes (daily, weekly, monthly, YTD, YoY)

Last update date

Data is collected from multiple HTML tables into a structured DataFrame

 Robust error handling ensures the scraper skips malformed rows safely.

In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_tradingeconomics_commodities():
    url = "https://tradingeconomics.com/commodities"
    headers = {
        "User-Agent": get_random_user_agent(),
        "Accept-Language": "en-US,en;q=0.9",
    }

    data = []

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        tables = soup.find_all("table", class_="table")

        for table in tables:
            thead = table.find("thead")
            tbody = table.find("tbody")

            if not thead or not tbody:
                continue

            # CATEGORY FROM FIRST <th>
            category = thead.find("th").get_text(strip=True)

            rows = tbody.find_all("tr")

            for row in rows:
                try:
                    commodity = row.find("td", class_="datatable-item-first").b.get_text(strip=True)
                    unit = row.find("td", class_="datatable-item-first").div.get_text(strip=True)

                    price = row.find("td", id="p").get_text(strip=True)
                    day_change = row.find("td", id="nch").get_text(strip=True)
                    pct_change = row.find("td", id="pch").get_text(strip=True)

                    tds = row.find_all("td")

                    weekly = tds[4].get_text(strip=True)
                    monthly = tds[5].get_text(strip=True)
                    ytd = tds[6].get_text(strip=True)
                    yoy = tds[7].get_text(strip=True)
                    date = row.find("td", id="date").get_text(strip=True)

                    data.append([
                        category, commodity, unit, price, day_change,
                        pct_change, weekly, monthly, ytd, yoy, date
                    ])

                except Exception:
                    continue

        columns = [
            "Category", "Commodity", "Unit", "Price", "Day_Change",
            "Pct_Change", "Weekly", "Monthly", "YTD", "YoY", "Date"
        ]

        return pd.DataFrame(data, columns=columns)

    except Exception as e:
        print("Error:", e)
        return None


# CALL FUNCTION
df = scrape_tradingeconomics_commodities()
print(df.head())
print(df["Category"].unique())



  Category    Commodity       Unit   Price Day_Change Pct_Change   Weekly  \
0   Energy    Crude Oil    USD/Bbl  58.785      1.025      1.77%    2.56%   
1   Energy        Brent    USD/Bbl  63.007      1.017      1.64%    3.72%   
2   Energy  Natural gas  USD/MMBtu  3.1553     0.2517     -7.39%  -12.79%   
3   Energy     Gasoline    USD/Gal  1.7768     0.0015     -0.08%    4.54%   
4   Energy  Heating Oil    USD/Gal  2.1292     0.0097      0.46%    0.67%   

   Monthly      YTD      YoY    Date  
0    0.56%    2.38%  -23.23%  Jan/09  
1    1.28%    3.55%  -21.00%  Jan/09  
2  -31.33%  -14.40%  -20.90%  Jan/09  
3   -0.64%    3.85%  -14.31%  Jan/09  
4   -6.33%    0.36%  -14.88%  Jan/09  
['Energy' 'Metals' 'Agricultural' 'Industrial' 'Livestock' 'Index'
 'Electricity']


## Initial Data Quality Checks

Inspect data types and column distributions

Count missing values

Analyze uniqueness per column

Validate overall data completeness

This ensures early detection of scraping or formatting issues.

In [6]:
# Display data types for each column
print("Data types:")
print(df.dtypes)

print("\nData type summary:")
print(f"  Numeric columns: {df.select_dtypes(include=['int64', 'float64']).columns.tolist()}")
print(f"  Text columns: {df.select_dtypes(include=['object']).columns.tolist()}")

Data types:
Category      object
Commodity     object
Unit          object
Price         object
Day_Change    object
Pct_Change    object
Weekly        object
Monthly       object
YTD           object
YoY           object
Date          object
dtype: object

Data type summary:
  Numeric columns: []
  Text columns: ['Category', 'Commodity', 'Unit', 'Price', 'Day_Change', 'Pct_Change', 'Weekly', 'Monthly', 'YTD', 'YoY', 'Date']


## Column Standardization

Converts column names to lowercase with underscores

Splits unit information into:

- currency

- measure

Improves consistency and downstream usability

In [7]:
# Display the number of unique values in each column
unique_values = df.nunique().sort_values(ascending=False)
print("Number of Unique Values per Column:")
print(unique_values)

Number of Unique Values per Column:
Commodity     99
Price         99
YoY           98
Monthly       92
YTD           91
Weekly        89
Day_Change    79
Pct_Change    77
Unit          37
Category       7
Date           5
dtype: int64


## Numeric Data Cleaning

Converts all numeric fields:

Removes commas and % symbols

Coerces invalid values to NaN

Columns cleaned include:

- Price

- Day change

- Percentage change

- Weekly / Monthly / YTD / YoY changes

In [8]:
# Calculate missing values
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

# Create summary DataFrame
missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Percentage': missing_pct
})

# Display only columns with missing values
missing_with_nulls = missing_df[missing_df['Missing Count'] > 0]

if len(missing_with_nulls) > 0:
    print("Missing values detected:")
    print(missing_with_nulls)
else:
    print("No missing values detected - excellent data quality!")

No missing values detected - excellent data quality!


In [9]:
# Store original column names for reference
original_columns = df.columns.tolist()

# Convert column names to lowercase and replace spaces with underscores
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Display the transformation
print("Column Name Transformation:")
for old, new in zip(original_columns, df.columns):
    print(f"{old:30} -> {new}")

Column Name Transformation:
Category                       -> category
Commodity                      -> commodity
Unit                           -> unit
Price                          -> price
Day_Change                     -> day_change
Pct_Change                     -> pct_change
Weekly                         -> weekly
Monthly                        -> monthly
YTD                            -> ytd
YoY                            -> yoy
Date                           -> date


In [ ]:
# Split the column into two parts by '/'
df[['currency', 'measure']] = df['unit'].str.split('/', expand=True)

# Check results
print(df[['unit', 'currency', 'measure']].head())

        unit currency measure
0    USD/Bbl      USD     Bbl
1    USD/Bbl      USD     Bbl
2  USD/MMBtu      USD   MMBtu
3    USD/Gal      USD     Gal
4    USD/Gal      USD     Gal


## Currency & Unit Normalization

Standardizes USD and cents-based pricing

Cleans mixed unit representations (e.g., USD/T, cents)

Ensures currency consistency for later conversions

In [ ]:
def normalize_unit_currency(df):
    # Ensure string type
    df["unit"] = df["unit"].astype(str).str.strip()
    df["currency"] = df["currency"].astype(str).str.strip()

    # unit contains USD/T
   
    mask_usd_t = df["unit"].str.contains(r"USD\s*/\s*T", case=False, na=False)

    df.loc[mask_usd_t, "unit"] = (
        df.loc[mask_usd_t, "unit"]
        .str.replace(r"USD\s*/\s*T", "", regex=True)
        .str.strip()
    )

    df.loc[mask_usd_t, "currency"] = "USD"

    # currency or unit mentions cents

    mask_cents = (
        df["currency"].str.contains("cent", case=False, na=False)
        | df["unit"].str.contains("cent", case=False, na=False)
    )

    df.loc[mask_cents, "currency"] = "USD"

    # Remove 'cents' text from unit if present
    df.loc[mask_cents, "unit"] = (
        df.loc[mask_cents, "unit"]
        .str.replace("cents?", "", case=False, regex=True)
        .str.strip()
    )

    # Final cleanup
    df["unit"] = df["unit"].replace("", None)

    return df



## Duplicate Handling

Removes duplicate commodity records

Uses business logic instead of row index:

Commodity

Currency

Date

Scrape date

Ensures latest data always replaces older entries.

In [12]:
# Check for duplicate titles
duplicate_titles = df['commodity'].duplicated().sum()

print(f"Duplicate commodities: {duplicate_titles}")

if duplicate_titles > 0:
    print("\nDuplicate commodities found:")
    print(df[df['commodity'].duplicated(keep=False)][['commodity', 'price', 'category']].sort_values('commodity'))

Duplicate commodities: 0


In [13]:
num_cols = ['price', 'day_change', 'pct_change', 'weekly', 'monthly', 'ytd', 'yoy']

for col in num_cols:
    # Remove commas and percent signs, convert to float
    df[col] = df[col].astype(str)             # ensure it is string first
    df[col] = df[col].str.replace(',', '')    # remove commas
    df[col] = df[col].str.replace('%', '')    # remove percent sign
    df[col] = pd.to_numeric(df[col], errors='coerce')  # convert to float, invalid => NaN

# Check
print(df[num_cols].dtypes)
print(df[num_cols].head())

price         float64
day_change    float64
pct_change    float64
weekly        float64
monthly       float64
ytd           float64
yoy           float64
dtype: object
     price  day_change  pct_change  weekly  monthly    ytd    yoy
0  58.7850      1.0250        1.77    2.56     0.56   2.38 -23.23
1  63.0070      1.0170        1.64    3.72     1.28   3.55 -21.00
2   3.1553      0.2517       -7.39  -12.79   -31.33 -14.40 -20.90
3   1.7768      0.0015       -0.08    4.54    -0.64   3.85 -14.31
4   2.1292      0.0097        0.46    0.67    -6.33   0.36 -14.88


In [14]:
# Display data types for each column
print("Data types:")
print(df.dtypes)

print("\nData type summary:")
print(f"  Numeric columns: {df.select_dtypes(include=['int64', 'float64']).columns.tolist()}")
print(f"  Text columns: {df.select_dtypes(include=['object']).columns.tolist()}")

Data types:
category       object
commodity      object
unit           object
price         float64
day_change    float64
pct_change    float64
weekly        float64
monthly       float64
ytd           float64
yoy           float64
date           object
currency       object
measure        object
dtype: object

Data type summary:
  Numeric columns: ['price', 'day_change', 'pct_change', 'weekly', 'monthly', 'ytd', 'yoy']
  Text columns: ['category', 'commodity', 'unit', 'date', 'currency', 'measure']


## Date Engineering

Converts partial dates (e.g., Dec/12) into full datetime objects

Appends the current year automatically

Extracts:

- Year

- Month

- Day

Adds a scrape_date timestamp for version tracking

In [ ]:
# Given the scraped DataFrame is df
# and the original date column is 'date' (values like 'Dec/12')

# Ensure column is string and strip any spaces
df['date_raw'] = df['date'].astype(str).str.strip()

# Add current year to make a full date and convert to datetime
current_year = pd.Timestamp.now().year
df['date_full'] = pd.to_datetime(df['date_raw'] + f'/{current_year}', format='%b/%d/%Y', errors='coerce')

# Step 3: Extract year, month, day
df['year'] = df['date_full'].dt.year
df['month'] = df['date_full'].dt.month
df['day'] = df['date_full'].dt.day

# Check results
print(df[['date', 'date_full', 'year', 'month', 'day']].head())


     date  date_full  year  month  day
0  Jan/09 2026-01-09  2026      1    9
1  Jan/09 2026-01-09  2026      1    9
2  Jan/09 2026-01-09  2026      1    9
3  Jan/09 2026-01-09  2026      1    9
4  Jan/09 2026-01-09  2026      1    9


In [ ]:
# Add scrape date column
df["scrape_date"] = pd.Timestamp.now().date()

In [18]:
# Display data types for each column
print("Data types:")
print(df.dtypes)

print("\nData type summary:")
print(f"  Numeric columns: {df.select_dtypes(include=['int64', 'float64']).columns.tolist()}")
print(f"  Text columns: {df.select_dtypes(include=['object']).columns.tolist()}")

Data types:
category               object
commodity              object
unit                   object
price                 float64
day_change            float64
pct_change            float64
weekly                float64
monthly               float64
ytd                   float64
yoy                   float64
date                   object
currency               object
measure                object
date_raw               object
date_full      datetime64[ns]
year                    int32
month                   int32
day                     int32
scrape_date            object
dtype: object

Data type summary:
  Numeric columns: ['price', 'day_change', 'pct_change', 'weekly', 'monthly', 'ytd', 'yoy']
  Text columns: ['category', 'commodity', 'unit', 'date', 'currency', 'measure', 'date_raw', 'scrape_date']


In [19]:
df = df.drop_duplicates()

## Currency Conversion to USD

Fetches real-time exchange rates using an API

Converts all prices into USD

Safely handles unsupported or missing currencies

    Creates a unified price_usd field for analysis.

In [20]:
# 1️ Fetch latest exchange rates with USD as the base
url = "https://api.exchangerate.host/latest?base=USD"
response = requests.get(url)
data = response.json()

# rates: currency -> USD conversion factor
rates = data.get("rates", {})  # safe get

# Function to convert a price to USD
def convert_price_to_usd(row):
    try:
        curr = row['currency']
        price = float(row['price']) 
        if curr == "USD":
            return price
        if curr in rates:
            # If base is USD, rate tells how many units of that currency = 1 USD.
            # To convert price in that currency to USD:
            return price / rates[curr]
        else:
            # If missing, just return price as is or None
            return None
    except Exception:
        return None

# Apply conversion
df['price_usd'] = df.apply(convert_price_to_usd, axis=1)

print(df[['commodity','unit','currency','price','price_usd']].head())


     commodity       unit currency    price  price_usd
0    Crude Oil    USD/Bbl      USD  58.7850    58.7850
1        Brent    USD/Bbl      USD  63.0070    63.0070
2  Natural gas  USD/MMBtu      USD   3.1553     3.1553
3     Gasoline    USD/Gal      USD   1.7768     1.7768
4  Heating Oil    USD/Gal      USD   2.1292     2.1292


## Dataset Versioning & Persistence

Merges new data with historical CSV data

Generates a stable unique ID per commodity per date

Deduplicates using business keys

Saves cleaned data to: "data/commodities_cleaned.csv"

In [21]:
os.makedirs('data', exist_ok=True)
cleaned_file = "data/commodities_cleaned.csv"

KEY_COLS = ["commodity", "unit", "currency", "date_full", "scrape_date"]

# Ensure scrape_date is datetime
df["scrape_date"] = pd.to_datetime(df["scrape_date"]).dt.date  # DATE ONLY

if os.path.isfile(cleaned_file):
    old_df = pd.read_csv(cleaned_file)
    old_df["scrape_date"] = pd.to_datetime(old_df["scrape_date"],format="mixed",errors="coerce").dt.date
else:
    old_df = pd.DataFrame()

# Combine
df_combined = pd.concat([old_df, df], ignore_index=True)
df = normalize_unit_currency(df)
# Deduplicate
df_combined = (
    df_combined
    .sort_values("scrape_date")
    .drop_duplicates(subset=KEY_COLS, keep="last")
    .reset_index(drop=True)
)

# Stable ID (business key)
df_combined["id"] = (
    df_combined["commodity"]
    + "_"
    + df_combined["currency"]
    + "_"
    + df_combined["date_full"].astype(str)
)

df_combined.to_csv(cleaned_file, index=False)

print(" commodities_cleaned.csv updated successfully")
print(f"Total rows: {len(df_combined)}")

#  Correct checks
print("IDs unique:", df_combined['id'].nunique() == len(df_combined))
print(df_combined[['id','commodity','date_full','scrape_date']].tail(10))


 commodities_cleaned.csv updated successfully
Total rows: 873
IDs unique: False
                                       id     commodity            date_full  \
863      Palm Oil_MYR_2026-01-09 00:00:00      Palm Oil  2026-01-09 00:00:00   
864        Lumber_USD_2026-01-09 00:00:00        Lumber  2026-01-09 00:00:00   
865         Wheat_USd_2026-01-09 00:00:00         Wheat  2026-01-09 00:00:00   
866      Soybeans_USd_2026-01-09 00:00:00      Soybeans  2026-01-09 00:00:00   
867      Titanium_CNY_2026-01-09 00:00:00      Titanium  2026-01-09 00:00:00   
868   Scrap Steel_USD_2026-01-08 00:00:00   Scrap Steel  2026-01-08 00:00:00   
869       Silicon_CNY_2026-01-09 00:00:00       Silicon  2026-01-09 00:00:00   
870  Orange Juice_USd_2026-01-09 00:00:00  Orange Juice  2026-01-09 00:00:00   
871         Spain_EUR_2026-01-09 00:00:00         Spain  2026-01-09 00:00:00   
872                                   NaN           NaN                  NaN   

    scrape_date  
863  2026-01-09  
864

In [22]:
for col in ["commodity", "unit", "category"]:
    print(col, df[col].str.len().max())

commodity 27
unit 19.0
category 12


A cleaned, deduplicated, currency-normalized, time-aware commodity dataset, ready for:

Power BI dashboards

Feature engineering

Machine learning modeling

Time-series analysis

## Conclusion
This pipeline successfully transforms raw web-scraped commodity market data into a high-quality, standardized analytical dataset. By combining robust scraping techniques, systematic cleaning, currency normalization, and versioned storage, the workflow ensures data reliability and reproducibility. The resulting dataset supports meaningful percentage-change modeling across commodities, currencies, and units—forming a strong foundation for visualization, predictive modeling, and decision-making.